# Linear hierarchical causal model: estimate vs reality

Generate data from a **linear** confounder HCM (Gaussian unit/subunit noise), compute the **true ATE** in closed form, and compare with the **per-unit linear regression** estimator (paper-style backdoor via units).

In [1]:
import numpy as np
from linear_hcm_data import (
    LinearConfounderHCM,
    default_linear_confounder_params,
    estimate_ate_linear_per_unit,
)

In [2]:
rng = np.random.default_rng(42)
params = default_linear_confounder_params(beta_a=1.0, beta_u=0.8, gamma_u=0.6)
model = LinearConfounderHCM(**params)
true_ate = model.true_ate()
print("True ATE (closed form):", round(true_ate, 4))

True ATE (closed form): 1.0


In [3]:
n, m = 200, 100
U, A, Y = model.sample(n, m, rng)
est_ate = estimate_ate_linear_per_unit(A, Y)
print("Estimated ATE (per-unit linear):", round(est_ate, 4))
print("|est - true|:", round(abs(est_ate - true_ate), 4))

Estimated ATE (per-unit linear): 0.9952
|est - true|: 0.0048


In [4]:
print("Multi-seed (n=200, m=100, 20 seeds):")
true_ate = model.true_ate()
errors = []
for seed in range(20):
    U, A, Y = model.sample(n, m, np.random.default_rng(seed))
    est = estimate_ate_linear_per_unit(A, Y)
    errors.append(est - true_ate)
errors = np.array(errors)
print("  bias (mean est - true):", round(float(np.mean(errors)), 4))
print("  RMSE:", round(float(np.sqrt(np.mean(errors**2))), 4))

Multi-seed (n=200, m=100, 20 seeds):
  bias (mean est - true): -0.0017
  RMSE: 0.0084
